## **Part IV: Subspace Diagnostics & Feature De-correlation (Analysis)**

### **Question 1: The Total Variance Illusion**

1.  **Physical Nature of `Sensor_4`:** A uniqueness value exceeding $98\%$ indicates that the data from `Sensor_4` shares almost no information with the rest of the multi-sensor grid. Physically, this means `Sensor_4` is completely decoupled from the actual structural and operational modes ($f_1, f_2$) driving the plant. It is recording almost pure, localized white noise, confirming an instrumentation error or an open electrical channel.
2.  **How Noise Tricks the PCA Engine:** PCA works by finding directions that maximize total global variance ($\text{Var} = \text{Shared Variance} + \text{Localized Noise}$). Because `Sensor_4` has a massive noise scale ($\sigma^2 \approx 2.0$), it introduces a large amount of localized variance into the system. PCA cannot distinguish between structured, shared variance and raw localized noise. To maximize global variance, it aligns its top principal components with this noisy channel, inflating the top eigenvalues and creating a false impression of significant structured components.
3.  **Risks in Automated Anomaly Frameworks:** If an engineer builds a plant monitoring framework relying solely on PCA, the automated detection boundaries will be distorted by the noise floor of the broken sensor. A severe internal structural failure on channels 1, 2, or 3 could be masked or minimized because the system's variance is dominated by the noise on `Sensor_4`. This leads to missed detections of critical failures or a high rate of false alarms driven entirely by sensor noise.

---

### **Question 2: Decoupling Structural Loading via Rotation**

1.  **Varimax Relaxation of the PCA Hierarchy:** Traditional PCA enforces a strict mathematical hierarchy where eigenvectors must capture maximum global variance in a descending order ($\lambda_1 > \lambda_2 > \dots$). This forces the components to be complex linear combinations that spread weights across all sensors.
    
    The Varimax rotation in Factor Analysis relaxes this strict variance-maximization rule. It performs an orthogonal rotation on the factor loading matrix to maximize the variance of the squared loadings down each column. This drives the system toward a **simple structure**, forcing individual sensors to map cleanly onto a single factor while zeroing out cross-coupling on others.
2.  **Operational Troubleshooting Value:** From an operator's standpoint, a raw PCA matrix is a confusing mix of overlapping sensor signals. If an anomaly occurs, it is difficult to isolate which physical section is failing.

    The rotated FA heatmap isolates these relationships: `Sensor_1` and `Sensor_2` map exclusively onto `Factor 1` (Primary structural load), while `Sensor_3` maps onto `Factor 2` (Secondary operational mode). If `Factor 1` spikes, the operator knows instantly to investigate the physical area monitored by sensors 1 and 2, dramatically reducing diagnostic time during a crisis.

---

### **Question 3: Determining Subspace Truncation ($k$)**

1.  **Behavior of the Mean $Q$ Statistic Curve:** * Transitioning from $k=1$ to $k=2$, the Mean $Q$ statistic drops sharply. This happens because the second component captures the secondary operational process ($f_2$), clearing out a massive chunk of structured error variance from the residual subspace.
    * Transitioning from $k=2$ to $k=3$, the curve flattens out completely, forming a distinct "elbow." Adding a third component yields almost no further reduction in residual error.
2.  **Identifying True Subspace Dimensionality:** The sharp drop followed by a flat elbow identifies **$k=2$** as the true hidden physical dimensionality of the asset. This pattern shows that the remaining information beyond $k=2$ is purely random, uninformative white noise that cannot be modeled.
3.  **Consequences of Choosing $k=3$:** If an engineer incorrectly sets the truncation boundary to $k=3$, they are forcing the system to treat the random, localized noise of `Sensor_4` as a valid structural component. This overfits the baseline model, pulling pure instrument noise into the "clean" principal subspace and degrading the reliability of the anomaly detection boundaries.

---

### **Question 4: Operational Trade-offs in System Health Monitoring**

#### **Robustness Comparison Matrix**
* **The PCA Strategy ($T^2$ + $Q$):** Vulnerable to localized instrument failure. A sudden electrical short or calibration drift on a single sensor will trigger a major spike in the $Q$ statistic, generating a false alarm that suggests structural damage.
* **The FA Strategy (Direct Latent Scores):** Highly robust. Because the generative framework models sensor-specific noise within the diagonal matrix $\boldsymbol{\Psi}$, it isolates localized errors from the shared physical factors.

#### **Final Engineering Selection**
The **Factor Analysis (FA) Strategy** is significantly more robust against a single sensor losing calibration or experiencing an electrical short.

**Justification:** When a sensor fails, its specific noise scale shoots up while its correlation with the rest of the grid drops to zero. The FA engine captures this change by driving that channel's **Uniqueness metric ($\varphi^2$) close to $100\%$**, shifting the error variance entirely into the diagonal noise floor matrix $\boldsymbol{\Psi}$. Because the latent factor scores are calculated using Thomson’s regression method ($\mathbf{f}_i = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$), the weight assigned to the failed, uncorrelated sensor is automatically downscaled. This prevents localized instrument faults from corrupting the core tracking layer, allowing the plant to continue running without triggering false emergency shutdowns.